# 🧬 Remedia — Reseptör Odaklı İlaç Keşif Pipeline'ı (tek notebook, Colab)

Bu notebook, tüm keşif akışını **baştan sona Google Colab'da** çalıştırır:
hedef protein → bağlanma cebi → tohum moleküller → yeni molekül üretimi →
**GNINA (GPU) docking** → ADMET → sıralama → görsel sonuç. **Hiç dosya
taşımak, hiç git senkronizasyonu yapmak yok.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehmetg06/Remedia/blob/main/notebooks/remedia_pipeline.ipynb)

### 📋 Nasıl kullanılır
1. Üstten **Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU (T4)** seç.
   **GNINA docking GPU olmadan çalışmaz — GPU ZORUNLUDUR.**
2. **Runtime ▸ Run all**. İlk **Hücre 0** (Miniconda kurulumu) **kernel'i yeniden
   başlatır** — bu NORMAL. Kernel yeniden başladıktan sonra **'Run all'ı TEKRAR
   çalıştır**; bu sefer Miniconda zaten kurulu olduğu için restart olmadan
   baştan sona sorunsuz akar.
3. Her hücrenin çıktısında **`✅`** işaretini gör; bir sonraki hücre bir
   öncekinin çıktısını (değişkenlerini) kullanır, sırayı bozma.

> **Parametreler:** UniProt ID, üretim yöntemi (füzyon/genetik/BRICS/random),
> molekül sayısı gibi seçimleri ilgili hücrelerin üstündeki **form kutularından**
> (dropdown/kaydırıcı) yapabilirsin. Çok satırlı SMILES listesi ise (elle tohum
> girmek istersen) kod içindeki üçlü-tırnaklı `MANUAL_SEEDS` değişkenine yapıştırılır.

> ℹ️ **Not:** Bu notebook fpocket'i conda ile kurar ve gerekli **conda Terms of
> Service'i otomatik kabul eder**. fpocket kurulamazsa docking kutusu proteinin
> geometrik merkezine kurulur (çalışır ama ideal değildir).


## 🔴 Hücre 0 — Miniconda kurulumu (⚠️ KERNEL YENİDEN BAŞLAR)
**Ne yapıyor:** fpocket'i conda ile kurabilmek için Colab'a Miniconda kurar
(Colab'da varsayılan conda yoktur). **condacolab kernel'i yeniden başlatır** —
bu beklenen, normal bir davranıştır.
**Nasıl ilerlenir:**
1. **Bu ilk hücreyi çalıştır** (ya da `Run all` — otomatik buraya kadar gelir).
2. Kernel yeniden başlayınca (*"Your session crashed / restarted"* mesajı NORMAL)
   endişelenme.
3. **`Run all`'ı TEKRAR çalıştır** — bu sefer Miniconda zaten kurulu olduğu için
   restart OLMAZ ve notebook baştan sona sorunsuz akar.
**Devam etmeden önce:** İkinci `Run all`'da bu hücrede
`✅ Miniconda zaten kurulu` yazısını gör; sonraki hücrelere geç.


In [ ]:
#@title 🔴 Hücre 0 — Miniconda kur (KERNEL YENİDEN BAŞLAR) { display-mode: "form" }
# ⚠️ BU HÜCRE KERNEL'İ YENİDEN BAŞLATIR (fpocket'i conda ile kurmak için gerekli).
# Kernel yeniden başlayınca 'Run all'ı TEKRAR çalıştır — 2. seferde restart OLMAZ,
# notebook baştan sona akar.
import sys, subprocess


def _conda_ready():
    # condacolab.check(): conda düzgün kuruluysa sessiz geçer, değilse hata verir.
    try:
        import condacolab
        condacolab.check()
        return True
    except Exception:
        return False


if _conda_ready():
    print("✅ Miniconda zaten kurulu — kernel RESTART GEREKMİYOR. Sonraki hücrelere devam et.")
else:
    print("• condacolab (Miniconda) kuruluyor... Bu işlem kernel'i YENİDEN BAŞLATACAK.")
    print("  → Restart sonrası 'Run all'ı TEKRAR çalıştır; bu sefer sorunsuz baştan sona akar.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "condacolab"], check=False)
    import condacolab
    condacolab.install()   # <-- KERNEL BURADA YENİDEN BAŞLAR


## 🔵 Hücre 1 — Kurulum
**Ne yapıyor:** GNINA (GPU'lu docking binary'si), **fpocket'i conda ile**
(Hücre 0'da kurulan Miniconda üzerinden — gerekli conda Terms of Service'i
otomatik kabul ederek), RDKit, meeko ve gerekli Python paketlerini kurar;
`src/` modüllerini import edebilmek için Remedia reposunu klonlayıp `src/`'yi
`sys.path`'e ekler; GPU'yu kontrol eder.
**Ne kadar sürer:** ~2–4 dakika (GNINA binary'si büyüktür, bir kere iner).
**Devam etmeden önce:** En sonda `✅ Kurulum tamam` yazısını ve GPU satırında
`bulundu ✔` gör. `⚠️ GPU BULUNAMADI` görürsen menüden GPU seçip bu hücreyi
TEKRAR çalıştır. fpocket kurulamazsa (`⚠️ fpocket KURULAMADI`) sorun değil —
docking kutusu geometrik merkeze kurulur, pipeline yine çalışır.


In [ ]:
import os, sys, stat, subprocess, urllib.request, shutil, datetime
from pathlib import Path

# --- 1) Remedia reposunu klonla (src/ modüllerini import edebilmek için) ----
REPO_URL = "https://github.com/mehmetg06/Remedia.git"
REPO_DIR = Path("Remedia")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    print("• Remedia reposu klonlandi (Remedia/)")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=False)
    print("• Remedia reposu guncel")

SRC_DIR  = (REPO_DIR / "src").resolve()
DATA_DIR = (REPO_DIR / "data").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f"• src/ import yoluna eklendi: {SRC_DIR}")

# --- 2) Python paketleri ----------------------------------------------------
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import rdkit  # noqa: F401
    print("• RDKit zaten kurulu")
except ImportError:
    print("• RDKit kuruluyor..."); pip_install("rdkit")

for mod, pkg in [("meeko", "meeko"), ("requests", "requests"),
                 ("pandas", "pandas"), ("tqdm", "tqdm")]:
    try:
        __import__(mod)
    except ImportError:
        print(f"• {pkg} kuruluyor..."); pip_install(pkg)

# --- 3) fpocket (baglanma cebi tespiti) — conda ile (apt DEGIL) -------------
if shutil.which("fpocket") is None:
    print("• fpocket kuruluyor (conda · bioconda)...")
    # Conda Terms of Service'i otomatik kabul et — yoksa CondaToSNonInteractiveError.
    # || true: ToS zaten kabullu/gereksizse script'i kirmasin.
    subprocess.run("conda tos accept --override-channels "
                   "--channel https://repo.anaconda.com/pkgs/main || true", shell=True)
    subprocess.run("conda tos accept --override-channels "
                   "--channel https://repo.anaconda.com/pkgs/r || true", shell=True)
    # fpocket'i bioconda'dan kur (Miniconda Hucre 0'da kuruldu).
    subprocess.run("conda install -y -c bioconda -c conda-forge fpocket "
                   ">/dev/null 2>&1 || true", shell=True)

if shutil.which("fpocket"):
    print("• fpocket: bulundu ✔", shutil.which("fpocket"))
else:
    print("⚠️ fpocket KURULAMADI — docking kutusu proteinin GEOMETRIK MERKEZINE"
          " kurulacak (Hucre 2). Bu ideal degil ama CALISIR; daha iyi pocket"
          " tespiti icin fpocket/conda kurulumunu kontrol et.")

# --- 4) GNINA GPU binary'si -------------------------------------------------
GNINA_PATH = "/usr/local/bin/gnina"
GNINA_URLS = [
    "https://github.com/gnina/gnina/releases/download/v1.3/gnina",
    "https://github.com/gnina/gnina/releases/download/v1.1/gnina",
    "https://github.com/gnina/gnina/releases/download/v1.0.3/gnina",
]
if os.path.exists(GNINA_PATH) and os.path.getsize(GNINA_PATH) > 1_000_000:
    print("• GNINA zaten indirilmis")
else:
    subprocess.run("apt-get -qq install -y libboost-all-dev >/dev/null 2>&1", shell=True)
    ok = False
    for url in GNINA_URLS:
        try:
            print(f"• GNINA indiriliyor: {url}")
            urllib.request.urlretrieve(url, GNINA_PATH)
            if os.path.getsize(GNINA_PATH) > 1_000_000:
                ok = True; break
        except Exception as e:
            print(f"  (bu surum alinamadi: {e})")
    if not ok:
        raise RuntimeError("GNINA binary'si indirilemedi — bu hucreyi tekrar calistir.")
    m = os.stat(GNINA_PATH).st_mode
    os.chmod(GNINA_PATH, m | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

ver = subprocess.run([GNINA_PATH, "--version"], capture_output=True, text=True)
print("• GNINA:", ((ver.stdout or "") + (ver.stderr or "")).strip().splitlines()[0] if (ver.stdout or ver.stderr) else "kuruldu")

# --- 5) GPU kontrolu --------------------------------------------------------
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode == 0 and "NVIDIA-SMI" in (gpu.stdout or ""):
    print("• GPU: bulundu ✔")
else:
    print("⚠️ GPU BULUNAMADI! Runtime > Change runtime type > GPU (T4) sec, "
          "sonra BU HUCREYI TEKRAR calistir. GNINA docking (Hucre 5) GPU ZORUNLU.")

# --- 6) Ortak yollar (tum hucreler bunlari kullanir) ------------------------
RUN_ID = datetime.datetime.now().strftime("run_%Y%m%d_%H%M%S")
RESULTS_DIR = Path("results") / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n✅ Kurulum tamam. RUN_ID={RUN_ID}  ·  sonuc klasoru: {RESULTS_DIR}")

## 🔵 Hücre 2 — Hedef: yapı indir + bağlanma cebi bul
**Ne yapıyor:** `UNIPROT_ID` için AlphaFold yapısını **REST API'den** indirir
(`fetch_structure.py` mantığı — sabit URL değil), `fpocket` ile en yüksek
druggability'li bağlanma cebini bulup merkezini otomatik hesaplar. fpocket
yoksa/başarısızsa proteinin **geometrik merkezine** düşer (çalışır ama ideal değil).
**Parametreler:** Aşağıdaki **form kutusundan** `UNIPROT_ID`'yi ve docking kutusu
boyutunu seçebilirsin (başka bir hedef denemek için tek yapman gereken bu).
**Ne kadar sürer:** ~30–60 saniye.
**Devam etmeden önce:** `✅ HEDEF HAZIR` satırında reseptör yolunu ve docking
kutusu merkezini (Å) gör.


In [ ]:
#@title 🔵 Hücre 2 parametreleri — hedef & docking kutusu { display-mode: "form" }
UNIPROT_ID = "P00918"  #@param {type:"string"}
#@markdown Örnek hedefler: `P00918` (Karbonik Anhidraz II), `P30405` (CypD).
BOX_DIM = 20  #@param {type:"slider", min:10, max:30, step:1}
#@markdown Docking kutusu bir küptür; kenar uzunluğu (Å).
BOX_SIZE = (float(BOX_DIM), float(BOX_DIM), float(BOX_DIM))

import fetch_structure, pocket_detection

# 1) AlphaFold yapisini REST API'den indir (sabit URL DEGIL)
pdb_path = fetch_structure.fetch_alphafold(UNIPROT_ID)
RECEPTOR = str(pdb_path)
print(f"• Reseptor yapisi: {RECEPTOR}")

# 2) fpocket ile en yuksek druggability'li cebi bul, merkezini hesapla
CENTER = None
if shutil.which("fpocket"):
    try:
        pocket = pocket_detection.best_druggable_pocket(pdb_path)
        CENTER = tuple(round(c, 3) for c in pocket["center"])
        print(f"• En iyi cep: Pocket {pocket['pocket_number']}  "
              f"druggability={pocket['druggability']}  hacim={pocket['volume']} A^3")
    except Exception as e:
        print(f"⚠️ fpocket cep bulamadi ({e}) — geometrik merkeze dusuluyor.")

# 3) Fallback: fpocket yoksa/basarisizsa tum atomlarin agirlik merkezi
if CENTER is None:
    xs = ys = zs = 0.0; n = 0
    for line in Path(pdb_path).read_text().splitlines():
        if line.startswith(("ATOM", "HETATM")):
            xs += float(line[30:38]); ys += float(line[38:46]); zs += float(line[46:54]); n += 1
    CENTER = (round(xs / n, 3), round(ys / n, 3), round(zs / n, 3))
    print("⚠️ fpocket kurulamadi/cep bulunamadi — docking kutusu proteinin GEOMETRIK")
    print("   MERKEZINE kuruldu. Bu ideal degil ama CALISIR; daha iyi pocket tespiti")
    print("   icin fpocket kurulumunu kontrol et (Hucre 1).")

print(f"\n✅ HEDEF HAZIR")
print(f"   reseptor = {RECEPTOR}")
print(f"   docking kutusu merkezi (A) = {CENTER}   ·   boyut = {BOX_SIZE}")


## 🔵 Hücre 3 — Tohum moleküller
**Ne yapıyor:** `known_ligands.py` ile ChEMBL/PubChem'den hedefe ait bilinen
inhibitörleri otomatik getirir. Bulamazsa (veya kendi setini denemek istersen)
`MANUAL_SEEDS` üçlü-tırnaklı metnindeki SMILES'leri kullanır — form widget'ı
YOK, çok satırlı girdi sorunu yaşamazsın.
**Ne kadar sürer:** ~5–15 saniye.
**Devam etmeden önce:** `✅ N geçerli tohum molekül hazır` satırını ve listelenen
tohumları gör.

In [ ]:
from known_ligands import fetch_known_ligands
from rdkit import Chem

# Elle molekul girmek istersen buraya yapistir (cok satirli SMILES listesi —
# form kutusu DEGIL, ucul-tirnakli string; boylece cok satir sorunu yasamazsin).
# ChEMBL bos donerse otomatik olarak bunlar kullanilir.
# Her satir: SMILES [isim].  # ile baslayan satirlar yok sayilir.
MANUAL_SEEDS = """
NS(=O)(=O)c1ccccc1                benzenesulfonamide
CC(=O)Nc1nnc(s1)S(N)(=O)=O        acetazolamide
"""

def parse_seed_text(text):
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        out.append(line.split()[0])
    return out

# 1) ChEMBL/PubChem'den otomatik getir
ligands, msg = fetch_known_ligands(UNIPROT_ID, max_results=5)
print(msg)

if ligands:
    seeds = [l["smiles"] for l in ligands]
    for l in ligands:
        print(f"  • {l['name']}: {l['smiles']}  ({l.get('activity', '')})")
else:
    seeds = parse_seed_text(MANUAL_SEEDS)
    print("• Elle girilen tohumlar (MANUAL_SEEDS) kullaniliyor:")
    for s in seeds:
        print("  •", s)

# Guvenlik agi + gecerlilik suzgeci
if not seeds:
    seeds = parse_seed_text(MANUAL_SEEDS) or ["NS(=O)(=O)c1ccccc1"]
seeds = [s for s in seeds if Chem.MolFromSmiles(s) is not None]

print(f"\n✅ {len(seeds)} gecerli tohum molekul hazir.")

## 🟣 Hücre 3.5 — REINVENT4 kurulumu (opsiyonel, sadece "pretrained" yöntemi için)
**Ne yapıyor:** [REINVENT4](https://github.com/MolecularAI/REINVENT4)'ü GitHub'dan klonlar, pip
ile kurar ve AstraZeneca'nın Zenodo'da yayınladığı **halka açık, önceden eğitilmiş**
"prior" ağırlığını (`reinvent.prior`) indirir. Bu, ilaç-benzeri moleküllerin genel
kimyasal "dilbilgisini" öğrenmiş bir sinir ağıdır — **herhangi bir reseptöre
yönlendirilmemiştir** ve **burada da yönlendirilmez**: hiçbir fine-tuning/RL/transfer
learning YAPILMAZ, yalnızca var olan prior'dan örnekleme ("sampling") yapılır.
Reseptöre uygunluk yine Hücre 5'teki GNINA docking + Hücre 6'daki ADMET filtresiyle
SONRADAN test edilir.

**Ne kadar sürer / ne kadar yer kaplar:** İlk çalıştırmada ~5 dakika, ~3-5 GB indirme
(PyTorch + CUDA kütüphaneleri dahil — Colab GPU ortamında normaldir). Kurulum **TEK
SEFERLİK**tir: `Run all`'ı tekrar çalıştırdığında (ör. kernel restart sonrası) zaten
kuruluysa hiçbir adım tekrarlanmaz, hücre saniyeler içinde biter.

**Atlamak istersen:** Aşağıdaki `INSTALL_REINVENT4` kutusunu kapat — yalnızca Hücre
4'te `pretrained` yöntemini SEÇMEYECEKSEN bunu güvenle atlayabilirsin (diğer dört
yöntem buna hiç ihtiyaç duymaz). `pretrained`'i yine de seçersen, Hücre 4 kurulumu
otomatik olarak (bu hücreyi atlamış olsan bile) kendi kendine tamamlar.


In [ ]:
#@title 🟣 Hücre 3.5 parametreleri — REINVENT4 kurulumu { display-mode: "form" }
INSTALL_REINVENT4 = True  #@param {type:"boolean"}
#@markdown Kapatırsan bu hücre atlanır (yalnızca `pretrained` yöntemini Hücre 4'te
#@markdown SEÇMEYECEKSEN güvenle kapatabilirsin).

if INSTALL_REINVENT4:
    from generative_model import install_reinvent
    install_reinvent(log_fn=print)
else:
    print("⏭️  REINVENT4 kurulumu atlandi (INSTALL_REINVENT4=False). "
          "'pretrained' yontemini secersen Hucre 4 otomatik kurar.")


## 🔵 Hücre 4 — Molekül üret (yöntem seç: füzyon / genetik / BRICS / random / pretrained)
**Ne yapıyor:** Aşağıdaki **form kutusundan** yöntemi seçebilirsin:
- **fusion** — dört aşamalı füzyon motoru (geniş keşif → ön eleme → genetik opt.
  → rafinasyon). Varsayılan ve en kapsamlı. `molecule_generator.py`, tohum gerektirir.
- **genetic** — saf genetik algoritma (nesil sayısını kaydırıcıdan ayarla). `molecule_generator.py`, tohum gerektirir.
- **brics** — BRICS fragman rekombinasyonu. `molecule_generator.py`, tohum gerektirir.
- **random** — rastgele mutasyon. `molecule_generator.py`, tohum gerektirir.
- **pretrained** — `generative_model.py` ile **REINVENT4**'ün hazır, önceden eğitilmiş
  prior modelinden örnekleme. **Tohum molekül GEREKTİRMEZ** (Hücre 3'teki tohumlar bu
  yöntemde kullanılmaz) — sıfırdan, öğrendiği kimyasal "dilbilgisinden" üretir.
  Reseptöre özel HİÇBİR eğitim yapılmaz; reseptöre uygunluk yine sonraki hücrelerde
  (GNINA docking + ADMET) test edilir. İlk kullanımda Hücre 3.5 çalışmadıysa REINVENT4'ü
  burada otomatik kurar (birkaç dakika sürebilir).

fusion/genetic/brics/random için üretim sırasında GA fitness'i olarak docking yerine
QED (ilaç-benzerlik) vekilini kullanır (`docking_opts=None`) — **gerçek docking bir
sonraki hücrede GNINA/GPU ile ayrı yapılır**, çünkü Vina kurulu değildir.
**Ne kadar sürer:** ~30–90 saniye (fusion/genetic/brics/random, CPU'da); `pretrained`
kurulumu ilk seferde birkaç dakika sürebilir, örnekleme kendisi saniyeler sürer.
**Devam etmeden önce:** `✅ N molekül üretildi` satırını ve örnek molekülleri gör.
Bu moleküller (`molecules`) sonraki hücrelerde docklanır.


In [ ]:
#@title 🔵 Hücre 4 parametreleri — üretim yöntemi & molekül sayısı { display-mode: "form" }
method = "fusion"  #@param ["fusion", "genetic", "brics", "random", "pretrained"]
GENERATE_N = 20  #@param {type:"slider", min:5, max:100, step:5}
#@markdown `GENERATE_N`: docking'e girecek hedef molekül sayısı.
GA_GENERATIONS = 5  #@param {type:"slider", min:2, max:20, step:1}
#@markdown `GA_GENERATIONS`: yalnızca `genetic` yönteminde nesil sayısı.
#@markdown `pretrained` (REINVENT4) tohum molekül SORMAZ — yalnızca yukarıdaki
#@markdown `GENERATE_N` kullanılır.

from molecule_generator import (
    fusion_generation, genetic_algorithm, random_mutation,
    brics_recombination, write_smi,
)

# docking_opts=None -> uretim sirasinda GA fitness'i QED vekilidir (Vina gerekmez);
# GERCEK docking bir sonraki hucrede GNINA/GPU ile ayrica yapilir.
print(f"⚙️  Yontem: {method}  ·  hedef molekul sayisi: {GENERATE_N}\n")

if method == "fusion":
    final, mode = fusion_generation(seeds, docking_opts=None, log_fn=print)
    generated = [smi for smi, _ in final]
elif method == "genetic":
    final, mode = genetic_algorithm(
        seeds,
        generations=int(GA_GENERATIONS),
        population_size=max(int(GENERATE_N), 15),
        docking_opts=None,
        log_fn=print,
    )
    generated = [smi for smi, _ in final]
elif method == "brics":
    generated = brics_recombination(seeds, n=int(GENERATE_N))
elif method == "random":
    generated = random_mutation(seeds, n=int(GENERATE_N))
elif method == "pretrained":
    # Tohum molekul GEREKTIRMEZ: REINVENT4'un onceden egitilmis, receptore ozel
    # OLMAYAN prior'undan sifirdan orneklenir. Receptore uygunluk sonraki
    # hucrelerde (GNINA + ADMET) test edilir.
    from generative_model import generate_with_reinvent
    smi_out = RESULTS_DIR / "generated.smi"
    generated = generate_with_reinvent(
        num_molecules=int(GENERATE_N),
        output_path=smi_out,
        prefix="mol",
        log_fn=print,
    )
else:
    raise ValueError(f"Bilinmeyen yontem: {method}")

# Havuz GENERATE_N'den kucukse (ozellikle fusion top-5 doner) kesif molekulleriyle
# tamamla. NOT: pretrained icin bu YAPILMAZ -- o yontem tohumsuz calisir, kucuk
# kalirsa REINVENT4'ten daha fazla ornek istemek gerekir (asagida REINVENT4 zaten
# hedeften %30 fazla orneklemistir); tohum bazli yontemlerle KARISTIRILMAZ.
if method != "pretrained" and len(generated) < GENERATE_N:
    pool = set(g for g in generated if g)
    pool.update(random_mutation(seeds, n=GENERATE_N))
    pool.update(brics_recombination(seeds, n=GENERATE_N))
    generated = [g for g in pool if g]
generated = generated[:GENERATE_N]

# (isim, SMILES) listesi — tum sonraki hucreler bunu kullanir
molecules = [(f"mol_{i:03d}", smi) for i, smi in enumerate(generated, 1)]

# Kayit icin .smi yaz (molecule_generator.write_smi formatinda) -- pretrained
# generate_with_reinvent() icinde zaten kendi smi_out'unu yazdi.
if method != "pretrained":
    smi_out = RESULTS_DIR / "generated.smi"
    write_smi(generated, smi_out, prefix="mol")

print(f"\n✅ {len(molecules)} molekul uretildi ({method})  ->  {smi_out}")
for name, smi in molecules[:10]:
    print(f"  {name}: {smi}")


## 🔵 Hücre 5 — GNINA DOCKING (GPU)
**Ne yapıyor:** Üretilen her molekülü RDKit ile 3D'ye hazırlar (`ligand_prep.py`
mantığı), sonra **GNINA'yı GPU'da CNN rescoring** ile çalıştırıp bağlanma
enerjisini (affinity, kcal/mol) hesaplar. Çıktı: `ligand, affinity_kcal_mol`
sütunlu bir DataFrame (`docking_df`).
**Hız ayarları (ilk tarama — hızlı olması amaçlanır):** `--cnn_scoring rescore`
(en hafif CNN modu), `--cnn fast` (varsayılan 3 modelli ensemble yerine TEK
hafif model), `--exhaustiveness 8` ve `--num_modes 1` (yalnızca en iyi poz
kullanılıyor, fazlası üretilmiyor). En iyi adaylar zaten `validate_top_candidates.py`
ile daha yüksek exhaustiveness'te yeniden dockleniyor — ilk taramada buna gerek yok.
**Ne kadar sürer:** molekül başına birkaç saniye ile birkaç dakika arası
değişebilir (protein/ligand boyutuna ve GPU'ya bağlı) — her molekül bittiğinde
gerçek geçen süre yazdırılır, bu yüzden kalan süreyi kendi ölçtüğün rakamdan
tahmin edebilirsin. Zaman aşımı YOKTUR — molekül ne kadar sürerse sürsün
docking tamamlanana kadar beklenir.
**Devam etmeden önce:** Her molekül için `✅ [i/N] isim: X kcal/mol (Y saniye)`
gör. Bir molekül `❌` verirse sorun değil — boş skorla raporlanır, diğerleri devam eder.
Daha negatif affinity = daha güçlü bağlanma.

In [ ]:
import re, time, subprocess
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm.auto import tqdm

DOCK_DIR = RESULTS_DIR / "gnina_out"
DOCK_DIR.mkdir(parents=True, exist_ok=True)


def prepare_ligand_sdf(smiles, name):
    # ligand_prep.py ile AYNI mantik: RDKit 3D + MMFF optimize, SDF yaz.
    # GNINA SDF'i dogrudan okur (Vina'daki ayri PDBQT adimina gerek yok).
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, randomSeed=42) != 0:
        return None
    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception:
        pass
    sdf = DOCK_DIR / f"{name}.sdf"
    w = Chem.SDWriter(str(sdf)); w.write(mol); w.close()
    return sdf


def parse_affinity(out_sdf, stdout):
    # En iyi (1. siradaki) pozun affinity'sini kcal/mol dondurur:
    # once cikti SDF property'lerine, olmazsa stdout tablosuna bakar.
    try:
        for m in Chem.SDMolSupplier(str(out_sdf), removeHs=False):
            if m is None:
                continue
            for key in ("minimizedAffinity", "CNNaffinity", "affinity"):
                if m.HasProp(key):
                    return float(m.GetProp(key))
            break
    except Exception:
        pass
    for line in stdout.splitlines():
        mt = re.match(r"\s*1\s+(-?\d+\.\d+)", line)
        if mt:
            return float(mt.group(1))
    return None


rows = []
n = len(molecules)
print(f"{n} molekul GNINA (GPU · CNN rescoring) ile docklanacak.\n")
start = time.time()
bar = tqdm(molecules, desc="GNINA", unit="mol")
for idx, (name, smiles) in enumerate(bar, 1):
    sdf = prepare_ligand_sdf(smiles, name)
    if sdf is None:
        print(f"❌ {name}: gecersiz SMILES / 3D uretilemedi — atlandi")
        rows.append({"ligand": name, "affinity_kcal_mol": None})
        continue

    out_sdf = DOCK_DIR / f"{name}_docked.sdf"
    # --cnn fast: yuksek verimli tarama icin TEK hafif CNN modeli kullanir
    # (varsayilan davranis, model belirtilmezse 3 modelli ensemble calistirir
    # -- rescore modunda bile ayni pozu 3 kez CNN'den gecirir, ~3x yavas).
    # --exhaustiveness 8: ilk taramada varsayilan (dogrulama asamasinda
    # validate_top_candidates.py zaten 32 ile yeniden dockluyor).
    # --num_modes 1: parse_affinity() zaten yalnizca en iyi (1.) pozu okuyor,
    # fazladan poz uretip CNN ile yeniden siralamaya gerek yok.
    cmd = [
        GNINA_PATH, "-r", RECEPTOR, "-l", str(sdf),
        "--center_x", str(CENTER[0]), "--center_y", str(CENTER[1]), "--center_z", str(CENTER[2]),
        "--size_x", str(BOX_SIZE[0]), "--size_y", str(BOX_SIZE[1]), "--size_z", str(BOX_SIZE[2]),
        "--cnn_scoring", "rescore", "--cnn", "fast",
        "--exhaustiveness", "8", "--num_modes", "1",
        "--seed", "42", "-o", str(out_sdf),
    ]
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    sure = time.time() - t0
    aff = parse_affinity(out_sdf, proc.stdout)

    if aff is not None:
        print(f"✅ [{idx}/{n}] {name}: {aff:.2f} kcal/mol  ({sure:.1f} saniye)")
    else:
        tail = (proc.stderr or proc.stdout or "").strip().splitlines()
        neden = ' | '.join(tail[-2:]) if tail else 'bilinmeyen hata'
        print(f"❌ [{idx}/{n}] {name}: docking basarisiz ({sure:.1f} saniye) — {neden}")
    rows.append({"ligand": name, "affinity_kcal_mol": aff})

    kalan = (time.time() - start) / idx * (n - idx) / 60.0
    bar.set_postfix_str(f"{idx}/{n} · tahmini kalan: {kalan:.1f} dk")

docking_df = pd.DataFrame(rows, columns=["ligand", "affinity_kcal_mol"])
ok = int(docking_df["affinity_kcal_mol"].notna().sum())
print(f"\n✅ GNINA bitti: {ok}/{n} molekul skorlandi.")
docking_df

## 🔵 Hücre 6 — ADMET (Lipinski / Veber)
**Ne yapıyor:** `admet_filter.py`'nin `lipinski_veber_filter`'ı ile her molekülün
MW, LogP, HBD/HBA, dönebilir bağ ve TPSA değerlerini hesaplayıp ilaç-benzerlik
filtresini uygular. Çıktı: `admet_df` DataFrame'i (`pass` sütunu geçti/kaldı).
**Ne kadar sürer:** ~1–5 saniye (offline, RDKit).
**Devam etmeden önce:** `✅ ... filtresini geçti` özet satırını gör.

In [ ]:
import admet_filter
import pandas as pd

admet_rows = [admet_filter.lipinski_veber_filter(smi, name) for name, smi in molecules]
admet_df = pd.DataFrame(admet_rows)

passed = int(admet_df["pass"].sum())
print(f"✅ ADMET: {passed}/{len(admet_df)} molekul Lipinski/Veber filtresini gecti.")
admet_df

## 🔵 Hücre 7 — Sırala (rank_report.py ile birleşik sıralama)
**Ne yapıyor:** Docking skorlarını ve ADMET sonuçlarını `rank_report.py`'nin
beklediği CSV formatlarında yazar, sonra `src/rank_report.py`'yi doğrudan çağırıp
birleşik sıralamayı üretir (önce ADMET'i geçenler, sonra en negatif affinity).
Çıktı: `ranked_df` ve `results/<RUN_ID>/final_ranking.csv`.
**Ne kadar sürer:** ~2 saniye.
**Devam etmeden önce:** `✅ Birleşik sıralama hazır` satırını ve sıralı tabloyu gör.

In [ ]:
import subprocess, sys
import pandas as pd

docking_csv = RESULTS_DIR / "docking_scores.csv"
admet_csv   = RESULTS_DIR / "admet_results.csv"
ranked_csv  = RESULTS_DIR / "final_ranking.csv"

# rank_report.py'nin okuyacagi tam formatlarda yaz
docking_df.to_csv(docking_csv, index=False)   # ligand, affinity_kcal_mol
admet_df.to_csv(admet_csv, index=False)       # ligand, pass, MW, LogP, violations, ...

# src/rank_report.py'yi dogrudan cagir
subprocess.run([
    sys.executable, str(SRC_DIR / "rank_report.py"),
    "--docking", str(docking_csv),
    "--admet", str(admet_csv),
    "--output", str(ranked_csv),
], check=True)

ranked_df = pd.read_csv(ranked_csv)
print(f"\n✅ Birlesik siralama hazir  ->  {ranked_csv}")
ranked_df

## 🔵 Hücre 8 — Sonuç: tablo + 2D çizim + Google Drive'a kaydet
**Ne yapıyor:** En iyi adayları tablo ve **RDKit 2D çizimleriyle** gösterir; tüm
sonuç dosyalarını (üretilen SMILES, docking/ADMET/sıralama CSV'leri, çizim PNG'si)
mount edilmiş **Google Drive**'da tarihli bir klasöre kaydeder — oturum kapansa
bile sonuçlar durur.
**Ne kadar sürer:** ~10–20 saniye (Drive mount izni ister).
**Devam etmeden önce:** En iyi aday tablosunu ve molekül çizimlerini gör; `✅ Tüm
sonuçlar Google Drive'a kaydedildi` satırını gör. Drive'a izin vermezsen sonuçlar
yine oturum içindeki `results/<RUN_ID>/` klasöründedir.

In [ ]:
import shutil
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

TOP_K = min(6, len(ranked_df))
top = ranked_df.head(TOP_K).copy()
print("🏆 EN IYI ADAYLAR:")
try:
    display(top)
except NameError:
    print(top.to_string(index=False))

# 2D cizimler — ligand ismini molecules uzerinden SMILES'e esle
smi_by_name = {name: smi for name, smi in molecules}
mols, legends = [], []
for _, r in top.iterrows():
    smi = smi_by_name.get(r["ligand"])
    m = Chem.MolFromSmiles(smi) if smi else None
    if m is None:
        continue
    aff = r["affinity_kcal_mol"]
    aff_s = f"{aff:.2f}" if pd.notna(aff) else "—"
    admet_ok = "✓" if bool(r.get("admet_pass", False)) else "✗"
    mols.append(m)
    legends.append(f"{r['ligand']}  {aff_s} kcal/mol  ADMET={admet_ok}")

grid_path = RESULTS_DIR / "top_candidates.png"
grid_saved = False
if mols:
    # Colab'da MolsToGridImage bazen PIL Image yerine IPython Image donduruyor
    # (useSVG/returnPNG davranisina gore); .save() olmayabilir, bu yuzden tipi
    # kontrol edip uygun kaydetme yolunu seciyoruz.
    grid = Draw.MolsToGridImage(mols, legends=legends, molsPerRow=3, subImgSize=(300, 250), returnPNG=False)
    try:
        if hasattr(grid, "save"):
            grid.save(str(grid_path))
            grid_saved = True
        elif hasattr(grid, "data"):
            with open(grid_path, "wb") as f:
                f.write(grid.data)
            grid_saved = True
        else:
            print(f"⚠️ Gorsel kaydedilemedi: bilinmeyen grid tipi ({type(grid)}).")
    except Exception as e:
        print(f"⚠️ Gorsel kaydedilemedi (pipeline devam ediyor): {e}")
    try:
        display(grid)
    except NameError:
        if grid_saved:
            print(f"(cizim kaydedildi: {grid_path})")

# --- Google Drive'a kalici kaydet (oturum kapansa bile durur) ---------------
saved = [docking_csv, admet_csv, ranked_csv, RESULTS_DIR / "generated.smi"]
if grid_saved and grid_path.exists():
    saved.append(grid_path)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = Path("/content/drive/MyDrive/Remedia_results") / RUN_ID
    save_dir.mkdir(parents=True, exist_ok=True)
    for f in saved:
        if Path(f).exists():
            shutil.copy(f, save_dir / Path(f).name)
    print(f"\n✅ Tum sonuclar Google Drive'a kaydedildi: {save_dir}")
except Exception as e:
    print(f"\nℹ️ Google Drive'a kaydedilemedi ({e}).")
    print(f"   Sonuclar oturum icinde hazir: {RESULTS_DIR.resolve()}")

print("\n🎉 PIPELINE TAMAMLANDI.")